# Chapter 38 — A Stochastic Step Is a Typed Function

**Companion to *Pi Agents*** · [`Pi Agents` chapter 38](https://programmer.ie/books/pi/38-chapter/)

| | |
|---|---|
| Chapter | `38-chapter.md` · weight 38 · `/books/pi/38-chapter/` |
| Notebook | `notebooks/pi/38-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Does `maxTurns` still bound a run **after** a valid submission was stored in a
mixed batch?

The chapter's point: a stochastic step is a **typed function** — input, schema,
bounds, and a typed value out. `maxAttempts` bounds repairs, `maxTurns` bounds
the run. The mixed-batch rule is the sharp edge: a submission in a batch with a
non-terminating tool cannot outlive `maxTurns`.


## What this notebook establishes

- A **valid submission** ends the run after one request and returns a typed value.
- An **invalid submission is returned to the model**, which repairs it, costing one request per attempt.
- **Repeated invalid submissions stop at `maxAttempts`**; a model that never submits is stopped by `maxTurns`.
- **Prose with no submit** is a `StepFailed` (kind `no-submit`), not a silent empty value.
- A **failed request** is a provider failure (kind `provider`), not the model choosing prose.
- In a **mixed batch**, a submission with a non-terminating tool cannot outlive `maxTurns` — the bound fires after a submission, not just before one.
- The **first valid submission wins** when the model submits more than once.
- An invalid submission in a mixed batch **costs one attempt, not one per tool**.

## What this notebook does **not** establish

- **`maxAttempts` and `maxTurns` are proposed bounds, not Pi's contract** (`ch38-lim2`).
- **Every case is a case someone anticipated** — a scripted test cannot show an autonomous surprise.
- **The model is scripted** — request counts are exact for the scripted control flow only.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — the chapter's `step()` against the scripted provider, all the boundary cases.
* **Evidence scope:** `in_process_runtime`. Schema validation, repair, and the batch rule are the chapter's step.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(38))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/38-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: valid, repair, and the bound

In [3]:
STEP = pinb.driver_source("ch38-typed-step.ts")
st = pinb.driver(STEP, label="ch38-typed-step", timeout=180)
v, r, a, n = st["valid"], st["repair"], st["attemptExhaustion"], st["neverSubmits"]
print("valid submission:   severity=" + str(v["out"]["severity"]) + f", requests={v["callCount"]}")
print("invalid then repair: severity=" + str(r["severity"]) + f", requests={r["callCount"]}")
print("attempt exhaustion:  kind=" + str(a["kind"]) + f", requests={a["callCount"]}")
print("never submits:       kind=" + str(n["kind"]) + f", requests={n["callCount"]}")
print()
pinb.show_checks([
    pinb.check("a valid submission ends the run after one request",
               v["out"]["severity"] == "high" and v["callCount"] == 1, "1 request"),
    pinb.check("an invalid submission is returned to the model, which repairs it",
               r["severity"] == "high" and r["callCount"] == 2, "2 requests"),
    pinb.check("repeated invalid submissions stop at maxAttempts",
               a["kind"] == "invalid-submissions" and a["callCount"] == 2, "2 requests then StepFailed"),
    pinb.check("a model that never submits is stopped by maxTurns",
               n["kind"] == "turn-limit" and n["callCount"] == 4, "4 requests then StepFailed"),
])

valid submission:   severity=high, requests=1
invalid then repair: severity=high, requests=2
attempt exhaustion:  kind=invalid-submissions, requests=2
never submits:       kind=turn-limit, requests=4

== Assertions ==
  assertion  observed
----  ----------------------------------------------------------------  --------------------------
PASS  a valid submission ends the run after one request  1 request
PASS  an invalid submission is returned to the model, which repairs it  2 requests
PASS  repeated invalid submissions stop at maxAttempts  2 requests then StepFailed
PASS  a model that never submits is stopped by maxTurns  4 requests then StepFailed

4/4 assertions held.


## The mixed-batch rule: the bound holds after a submission

In [4]:
mb, fv, ib = st["mixedBatchBound"], st["firstValidWins"], st["invalidInMixedBatch"]
print("submission + non-terminating tool: severity=" + str(mb["out"]["severity"]) + f", requests={mb["callCount"]}")
print("first valid wins:                  severity=" + str(fv["out"]["severity"]))
print("invalid in mixed batch:            severity=" + str(ib["severity"]) + f", requests={ib["callCount"]}")
print()
pinb.show_checks([
    pinb.check("a submission in a mixed batch cannot outlive maxTurns",
               mb["out"]["severity"] == "high" and mb["callCount"] == 2, "2 requests, bounded"),
    pinb.check("the first valid submission wins",
               fv["out"]["severity"] == "low", "first wins"),
    pinb.check("an invalid submission in a mixed batch costs one attempt, not one per tool",
               ib["severity"] == "high" and ib["callCount"] == 3, "3 requests, repaired"),
])

submission + non-terminating tool: severity=high, requests=2
first valid wins:                  severity=low
invalid in mixed batch:            severity=high, requests=3

== Assertions ==
  assertion  observed
----  --------------------------------------------------------------------------  --------------------
PASS  a submission in a mixed batch cannot outlive maxTurns  2 requests, bounded
PASS  the first valid submission wins  first wins
PASS  an invalid submission in a mixed batch costs one attempt, not one per tool  3 requests, repaired

3/3 assertions held.


## The chapter's own tests

All fourteen step tests.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [5]:
PATTERNS = 'ch38-typed-step/step.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch38-typed-step/step.test.ts', show="mixed")

$ node --test ch38-typed-step/step.test.ts
  PASS  14 passed, 0 failed, 0 skipped  in 399 ms

  tests matching 'mixed':
    ok   a submission in a mixed batch is still reported as a provider failure if the run then dies
    ok   an invalid submission in a mixed batch costs one attempt, not one per tool
    ok   invalid submissions in mixed batches still exhaust the attempt budget


## Your turn: predict, then run

**Predict first.** `maxTurns: 2` with a mixed batch stored a valid submission in
turn 1. What stops the loop in turn 2 — the valid value or the bound?

**Then change one input.** `maxAttempts: 3` with two invalid mixed batches that
each contain one invalid submission. How many requests before repair?

**Predict the boundary.** A submission in a batch with a tool that *does*
terminate. When does the turn end — immediately, or after the batch?

In [6]:
print("Predictions:")
print("  1. maxTurns=2 with a stored submission: the second turn must still find a terminating run.")
print("  2. Two invalid mixed batches: two attempts, then the third repairs - 3 requests.")
print("  3. A batch where every tool terminates: the turn ends when the batch agrees.")
print()
print("Observed above:")
vs = str(v["out"]["severity"])
rs = str(r["severity"])
mbs = str(mb["out"]["severity"])
ibs = str(ib["severity"])
print(f"  valid: severity={vs}, requests={v["callCount"]}")
print(f"  repair: severity={rs}, requests={r["callCount"]}")
print(f"  mixed batch bound: severity={mbs}, requests={mb["callCount"]}")
print(f"  invalid in mixed: severity={ibs}, requests={ib["callCount"]}")
print()
assert v["callCount"] == 1 and r["callCount"] == 2
assert a["kind"] == "invalid-submissions" and n["kind"] == "turn-limit"
assert mb["callCount"] == 2 and fv["out"]["severity"] == "low"
assert ib["callCount"] == 3
print("held: maxAttempts bounds repairs, maxTurns bounds the run, mixed batches cost one attempt")

Predictions:
  1. maxTurns=2 with a stored submission: the second turn must still find a terminating run.
  2. Two invalid mixed batches: two attempts, then the third repairs - 3 requests.
  3. A batch where every tool terminates: the turn ends when the batch agrees.

Observed above:
  valid: severity=high, requests=1
  repair: severity=high, requests=2
  mixed batch bound: severity=high, requests=2
  invalid in mixed: severity=high, requests=3

held: maxAttempts bounds repairs, maxTurns bounds the run, mixed batches cost one attempt


## Interpretation

A step is a typed function with three parts, and the chapter shows the bounds:

| Part | What it decides | The boundary |
|---|---|---|
| **Schema** | What counts as a valid submission | Invalid → returned to the model |
| **`maxAttempts`** | How many repairs | Exhausted → `StepFailed` kind `invalid-submissions` |
| **`maxTurns`** | How long the run can go | Exhausted → `StepFailed` kind `turn-limit` |

The mixed-batch rule: a turn ends early only when every completed result in the
batch agrees to terminate. `submit` agrees, `lookup_evidence` does not, so one
response can store a valid answer and still leave the loop running — the bound has
to keep firing after a submission, or the bound is not a bound.

Practical rules:
1. **Validate the submission** — the schema is the contract, and repairs cost requests.
2. **Bound both dimensions** — attempts limit repairs, turns limit the run.
3. **Count requests** — a valid step costs one; a repair costs two; a bounded run stops on the bound.
4. **Remember the batch rule** — a stored submission does not end the run unless the batch agrees.

## Sources, execution mode and limitations

**Execution mode:** **Run** — the chapter's `step()` against the scripted provider, all the boundary cases.

**Evidence scope:** `in_process_runtime`. Schema validation, repair, and the batch rule are the chapter's step.

### What was read or run

- **Ran** `drivers/ch38-typed-step.ts`: seven step scenarios (valid, repair, attempt exhaustion, never submits, mixed batch bound, first valid wins, invalid in mixed batch).
- **Ran** `ch38-typed-step/step.test.ts` (14 tests).
- **Read** `examples/evidence.json`, chapter 38 rows (14 claims).

### Limitations

- **`maxAttempts` and `maxTurns` are proposed bounds, not Pi's contract** (`ch38-lim2`).
- **Every case is a case someone anticipated** — a scripted test cannot show an autonomous surprise.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
